In [240]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

from pathlib import Path
import os
import sys
from datetime import datetime, date

import time
import itertools

MAX_CPU_TIME = 3600.0
EPSILON = 1e-6


In [241]:
result = Path('result/')
report = Path('report/')
instances = Path('../../data/csifa')

In [242]:
def read_instance(file_name):
	
	arq = open(file_name)
	
	N = int(arq.readline())
	
	PR  = [0]*N
	PP  = [0]*N

	FR = [float(arq.readline())]*N
	FP = [float(arq.readline())]*N

	HR = [float(arq.readline())]*N
	HP = [float(arq.readline())]*N

	D = [int(i) for i in arq.readline().split()]
	
	R = [int(i) for i in arq.readline().split()]

	C = float(arq.readline().rstrip('\n'))
	
	return N, PP, PR, FP, FR, HR, HP, D, R, C

In [243]:
def lsr_std_math_mip(N, PP, PR, FP, FR, HP, HR, D, R, SD, SR, yp_sol, yr_sol):

	yp_val = np.zeros(N)
	yr_val = np.zeros(N)

	try:

		# create model
		model = gp.Model("lsr_std_math_mip")

		# create variables
		xp = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="xp")
		xr = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="xr")
		sp = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="sp")
		sr = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="sr")
		yp = model.addVars(list(range(N)), lb=0.0, ub=1.0, vtype=GRB.BINARY, name="yp")
		yr = model.addVars(list(range(N)), lb=0.0, ub=1.0, vtype=GRB.BINARY, name="yr")

		# fix variables yp, yr
		#for i in range(N):
		#	yp[i].start = yp_sol[i]
		#	yr[i].start = yr_sol[i]
		#model.update()
		
		# set objective
		model.setObjective(
			gp.quicksum(PP[i]*xp[i] for i in range(N)) + 
			gp.quicksum(HP[i]*sp[i] for i in range(N)) + 
			gp.quicksum(FP[i]*yp[i] for i in range(N)) + 
			gp.quicksum(PR[i]*xr[i] for i in range(N)) + 
			gp.quicksum(HR[i]*sr[i] for i in range(N)) + 
			gp.quicksum(FR[i]*yr[i] for i in range(N)), sense = GRB.MINIMIZE)

		# add constraints
		model.addConstr(xp[0] + xr[0] - sp[0] == D[0])

		model.addConstrs(sp[i-1] + xp[i] + xr[i] - sp[i] == D[i] for i in range(N) if i > 0 )
		
		model.addConstr(R[0] - xr[0] - sr[0] == 0)
		
		model.addConstrs(sr[i-1] + R[i] - xr[i] - sr[i] == 0 for i in range(N) if i > 0)
		
		model.addConstrs(xp[i] - yp[i]*SD[i][N-1] <= 0 for i in range(N))
		
		model.addConstrs(xr[i] - yr[i]*min(SR[0][i], SD[i][N-1]) <= 0 for i in range(N))

		for i in range(2,N):
			for l in range(i,N):
				con = 0
				for k in range(i,l+1):
					con += SD[k][l]*(yp[k]+yr[k])
				model.addConstr(sp[i-1] + con >= SD[i][l])

		for i in range(1,N):
			for l in range(i,N):
				con = 0
				for k in range(i,l+1):
					con += SR[i][k]*yr[k]
				model.addConstr(sr[l] + con >= SR[i][l])
	   
	    #model.write("instance.lp")

		# set parameters 
		#model.setParam("LogFile", "gurobi_math.log")
		#model.setParam("LogToConsole", 0)
		model.setParam('OutputFlag', 0)

		model.setParam(GRB.Param.TimeLimit, MAX_CPU_TIME)
		model.setParam(GRB.Param.MIPGap, EPSILON)
		model.setParam(GRB.Param.Threads,1)
		#model.setParam(GRB.Param.Cuts, 0)
		#model.setParam(GRB.Param.Presolve,0)
		
		# relax model
		#for v in model.getVars():
		#	v.setAttr('vtype', 'C')

		# optimize model
		model.optimize()

		tmp = 0
		if model.status == GRB.OPTIMAL:
			tmp = 1

		objval = model.ObjVal
		objbound = model.ObjBound
		mipgap = model.MIPGap
		runtime = model.Runtime
		nodecount = model.NodeCount
	
	except gp.GurobiError as e:
		print('Error code ' + str(e.errno) + ': ' + str(e))

	return objval, objbound, mipgap, runtime, nodecount, tmp


In [244]:
def lower_bound(N, PP, PR, FP, FR, HP, HR, D, R, SD, SR, lambdap):

	#yp_val = np.zeros(N)
	#yr_val = np.zeros(N)

	try:

		# create model
		model = gp.Model("lsrp_lower_bound")

		# create variables
		xp = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="xp")
		xr = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="xr")
		sp = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="sp")
		sr = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="sr")
		yp = model.addVars(list(range(N)), lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="yp")
		yr = model.addVars(list(range(N)), lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="yr")
				
		# set objective
		fobj = 0
		fobj += gp.quicksum((PP[i]+lambdap[i])*xp[i] for i in range(N))
		fobj += gp.quicksum(HP[i]*sp[i] for i in range(N))
		fobj += gp.quicksum((FP[i]-lambdap[i]*SD[i][N-1])*yp[i] for i in range(N))
		fobj += gp.quicksum(PR[i]*xr[i] for i in range(N))
		fobj += gp.quicksum(HR[i]*sr[i] for i in range(N))
		fobj += gp.quicksum(FR[i]*yr[i] for i in range(N))
	
		model.setObjective(fobj, sense = GRB.MINIMIZE)

		# add constraints
		model.addConstr(xp[0] + xr[0] - sp[0] == D[0])

		model.addConstrs(sp[i-1] + xp[i] + xr[i] - sp[i] == D[i] for i in range(N) if i > 0 )
		
		model.addConstr(R[0] - xr[0] - sr[0] == 0)
		
		model.addConstrs(sr[i-1] + R[i] - xr[i] - sr[i] == 0 for i in range(N) if i > 0)
		
		#model.addConstrs(xp[i] - yp[i]*SD[i][N-1] <= 0 for i in range(N))
		
		model.addConstrs(xr[i] - yr[i]*min(SR[0][i], SD[i][N-1]) <= 0 for i in range(N))

		for i in range(2,N):
			for l in range(i,N):
				con = 0
				for k in range(i,l+1):
					con += SD[k][l]*(yp[k]+yr[k])
				model.addConstr(sp[i-1] + con >= SD[i][l])

		for i in range(1,N):
			for l in range(i,N):
				con = 0
				for k in range(i,l+1):
					con += SR[i][k]*yr[k]
				model.addConstr(sr[l] + con >= SR[i][l])
	   
	    #model.write("instance.lp")

		# set parameters 
		#model.setParam("LogFile", "gurobi_lower_bound.log")
		#model.setParam("LogToConsole", 0)
		model.setParam('OutputFlag', 0)

		model.setParam(GRB.Param.TimeLimit, MAX_CPU_TIME)
		model.setParam(GRB.Param.MIPGap, EPSILON)
		model.setParam(GRB.Param.Threads,1)
		#model.setParam(GRB.Param.Cuts, 0)
		#model.setParam(GRB.Param.Presolve,0)
		
		# relax model
		#for v in model.getVars():
		#	v.setAttr('vtype', 'C')

		# optimize model
		model.optimize()

		tmp = 0
		if model.status == GRB.OPTIMAL:
			tmp = 1

		objval = model.ObjVal

		xp_d = [xp[i].X for i in range(N)]
		yp_d = [yp[i].X for i in range(N)]
		sp_d = [sp[i].X for i in range(N)]
		xr_d = [xr[i].X for i in range(N)]
		yr_d = [yr[i].X for i in range(N)]
		sr_d = [sr[i].X for i in range(N)]	
	
	except gp.GurobiError as e:
		print('Error code ' + str(e.errno) + ': ' + str(e))

	return objval, xp_d, yp_d, sp_d, xr_d, yr_d, sr_d


In [245]:
def relax_fix(partp, partr, yp_sol , yr_sol, N, PP, PR, FP, FR, HP, HR, D, R, SD, SR):

	yp_val = np.zeros(N)
	yr_val = np.zeros(N)

	try:
		# create model
		model = gp.Model("lsr_rf")

		# create variables
		xp = model.addVars(list(range(N)), lb=0.0, ub=float('inf'), vtype=GRB.CONTINUOUS, name="xp")
		yp = model.addVars(list(range(N)), lb=0.0, ub=1.0, vtype=GRB.BINARY, name="yp")
		sp = model.addVars(list(range(N)), lb=0.0, ub=float('inf'), vtype=GRB.CONTINUOUS, name="sp")
		xr = model.addVars(list(range(N)), lb=0.0, ub=float('inf'), vtype=GRB.CONTINUOUS, name="xr")
		yr = model.addVars(list(range(N)), lb=0.0, ub=1.0, vtype=GRB.BINARY, name="yr")
		sr = model.addVars(list(range(N)), lb=0.0, ub=float('inf'), vtype=GRB.CONTINUOUS, name="sr")
		
		#if (partp is not None):
		#	print(f"partp_rf: {partp, max(partp)}")
			
		#if (partr is not None):
		#	print(f"partr_rf: {partr, max(partr)}")
			
		#input("\ntecle <enter> para continuar")
		
		for i in range(N):
			if (partp is not None) and (i > max(partp)):
				yp[i].VType = gp.GRB.CONTINUOUS
				yp[i].lb = 0.0
				yp[i].ub = 1.0
			elif (partp is not None) and (i in partp):
				yp[i].lb = 0.0
				yp[i].ub = 1.0
			else:
				yp[i].lb = yp_sol[i]
				yp[i].ub = yp_sol[i]

		for i in range(N):
			if (partr is not None) and (i > max(partr)):
				yr[i].VType = gp.GRB.CONTINUOUS
				yr[i].lb = 0.0
				yr[i].ub = 1.0
			elif (partr is not None) and (i in partr):
				yr[i].lb = 0.0
				yr[i].ub = 1.0
			else:
				yr[i].lb = yr_sol[i]
				yr[i].ub = yr_sol[i]
		
		model.update()

		# set objective
		fobj = gp.quicksum(PP[i]*xp[i] for i in range(N))
		fobj += gp.quicksum(HP[i]*sp[i] for i in range(N))
		fobj += gp.quicksum(FP[i]*yp[i] for i in range(N))
		fobj += gp.quicksum(PR[i]*xr[i] for i in range(N))
		fobj += gp.quicksum(HR[i]*sr[i] for i in range(N))
		fobj += gp.quicksum(FR[i]*yr[i] for i in range(N))
	
		model.setObjective(fobj, sense = GRB.MINIMIZE)

		# add constraints
		model.addConstr(xp[0] + xr[0] - sp[0] == D[0])
		
		model.addConstrs(sp[i-1] + xp[i] + xr[i] - sp[i] == D[i] for i in range(N) if i > 0 )
		
		model.addConstr(R[0] - xr[0] - sr[0] == 0)
		
		model.addConstrs(sr[i-1] + R[i] - xr[i] - sr[i] == 0 for i in range(N) if i > 0)
		
		model.addConstrs(xp[i] - yp[i]*SD[i][N-1] <= 0 for i in range(N))
		
		model.addConstrs(xr[i] - yr[i]*min(SR[0][i], SD[i][N-1]) <= 0 for i in range(N))

		for i in range(2,N):
			for l in range(i,N):
				con = 0
				for k in range(i,l+1):
					con += SD[k][l]*(yp[k]+yr[k])
				model.addConstr(sp[i-1] + con >= SD[i][l])

		for i in range(1,N):
			for l in range(i,N):
				con = 0
				for k in range(i,l+1):
					con += SR[i][k]*yr[k]
				model.addConstr(sr[l] + con >= SR[i][l])
				
		#model.write("instance.lp")

		# set parameters 
		#model.setParam("LogFile", "gurobi_rf.log")
		#model.setParam("LogToConsole", 0)
		model.setParam('OutputFlag', 0)

		model.setParam(GRB.Param.TimeLimit, MAX_CPU_TIME)
		model.setParam(GRB.Param.MIPGap, EPSILON)
		model.setParam(GRB.Param.Threads,1)
		#model.setParam(GRB.Param.Cuts,0)
		#model.setParam(GRB.Param.Presolve,0)
		
		# Optimize model
		model.optimize()

		objval = model.ObjVal

		yp_val = [yp[i].X for i in range(N)]
		yr_val = [yr[i].X for i in range(N)]

	except gp.GurobiError as e:
		print('Error code ' + str(e.errno) + ': ' + str(e))

	return objval, yp_val, yr_val


In [246]:
def fix_and_optimize(partp, partr, yp_sol, yr_sol, N, PP, PR, FP, FR, HP, HR, D, R, SD, SR):

    try:
        yp_val = np.zeros(N)
        yr_val = np.zeros(N)

        # create model
        model = gp.Model("lsr_fo")

        # create variables
        xp = model.addVars(list(range(N)), lb =0.0, ub = float('inf'),vtype=GRB.CONTINUOUS, name="xp")
        yp = model.addVars(list(range(N)), lb =0.0, ub = 1.0,vtype=GRB.BINARY, name="yp")
        sp = model.addVars(list(range(N)), lb =0.0, ub = float('inf'),vtype=GRB.CONTINUOUS, name="sp")
        xr = model.addVars(list(range(N)), lb =0.0, ub = float('inf'),vtype=GRB.CONTINUOUS, name="xr")
        yr = model.addVars(list(range(N)), lb =0.0, ub = 1.0,vtype=GRB.BINARY, name="yr")
        sr = model.addVars(list(range(N)), lb =0.0, ub = float('inf'),vtype=GRB.CONTINUOUS, name="sr")

        #if (partp is not None):
        #    print(f"partp_fo: {partp, max(partp)}")
			
        #if (partr is not None):
        #    print(f"partr_fo: {partr, max(partr)}")
			
        #input("\ntecle <enter> para continuar")

        for i in range(N):
            if (partp is not None) and (i not in partp):
                yp[i].lb = yp_sol[i]
                yp[i].ub = yp_sol[i]
            else:
                yp[i].start = yp_sol[i]

        for i in range(N):
            if (partr is not None) and (i not in partr):
                yr[i].lb = yr_sol[i]
                yr[i].ub = yr_sol[i]
            else:
                yr[i].start = yr_sol[i]
                
        #for i in range(N):
        #    xp[i].start = xp_sol[i]
        #    xr[i].start = xr_sol[i]
        #    sp[i].start = sp_sol[i]
        #    sr[i].start = sr_sol[i]
        
        model.update()
        
        # set objective
        fobj = gp.quicksum(PP[i]*xp[i] for i in range(N))
        fobj += gp.quicksum(HP[i]*sp[i] for i in range(N))
        fobj += gp.quicksum(FP[i]*yp[i] for i in range(N))
        fobj += gp.quicksum(PR[i]*xr[i] for i in range(N))
        fobj += gp.quicksum(HR[i]*sr[i] for i in range(N))
        fobj += gp.quicksum(FR[i]*yr[i] for i in range(N))
        
        model.setObjective(fobj, sense = GRB.MINIMIZE)

        # add constraints
        model.addConstr(xp[0] + xr[0] - sp[0] == D[0])
        
        model.addConstrs(sp[i-1] + xp[i] + xr[i] - sp[i] == D[i] for i in range(N) if i > 0 )
        
        model.addConstr(R[0] - xr[0] - sr[0] == 0)
        
        model.addConstrs(sr[i-1] + R[i] - xr[i] - sr[i] == 0 for i in range(N) if i > 0)
        
        model.addConstrs(xp[i] - yp[i]*SD[i][N-1] <= 0 for i in range(N))
        
        model.addConstrs(xr[i] - yr[i]*min(SR[0][i], SD[i][N-1]) <= 0 for i in range(N))

        for i in range(2,N):
            for l in range(i,N):
                con = 0
                for k in range(i,l+1):
                    con += SD[k][l]*(yp[k]+yr[k])
                model.addConstr(sp[i-1] + con >= SD[i][l])

        for i in range(1,N):
            for l in range(i,N):
                con = 0
                for k in range(i,l+1):
                    con += SR[i][k]*yr[k]
                model.addConstr(sr[l] + con >= SR[i][l])
               
        #model.write("instance.lp")

        # set parameters 
        #model.setParam("LogFile", "gurobi_fo.log")
        #model.setParam("LogToConsole", 0)
        model.setParam('OutputFlag', 0)

        model.setParam(GRB.Param.TimeLimit, MAX_CPU_TIME)
        model.setParam(GRB.Param.MIPGap, EPSILON)
        model.setParam(GRB.Param.Threads,1)
        #model.setParam(GRB.Param.Cuts,0)
        #model.setParam(GRB.Param.Presolve,0)
        
        # optimize model
        model.optimize()

        xp_val = [xp[i].X for i in range(N)]
        xr_val = [xr[i].X for i in range(N)]
        yp_val = [yp[i].X for i in range(N)]
        yr_val = [yr[i].X for i in range(N)]
        sp_val = [sp[i].X for i in range(N)]
        sr_val = [sr[i].X for i in range(N)]

        objval = model.ObjVal

    except gp.GurobiError as e:
        print('Error code ' + str(e.errno) + ': ' + str(e))

    return objval, xp_val, xr_val, yp_val, yr_val, sp_val, sr_val


In [247]:
def clsr_std_mip(N, PP, PR, FP, FR, HR, HP, D, R, SD, SR, C):
	try:

		# Create a new model
		model = gp.Model("clsr_std_mip")

		# Create variables
		xp = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="xp")
		yp = model.addVars(list(range(N)), vtype=GRB.BINARY, name="yp")
		sp = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="sp")
		xr = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="xr")
		yr = model.addVars(list(range(N)), vtype=GRB.BINARY, name="yr")
		sr = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="sr")
		
		model.update()

		# set objective
		fobj = gp.quicksum(PP[i]*xp[i] for i in range(N))
		fobj += gp.quicksum(HP[i]*sp[i] for i in range(N))
		fobj += gp.quicksum(FP[i]*yp[i] for i in range(N))
		fobj += gp.quicksum(PR[i]*xr[i] for i in range(N))
		fobj += gp.quicksum(HR[i]*sr[i] for i in range(N))
		fobj += gp.quicksum(FR[i]*yr[i] for i in range(N))
	
		model.setObjective(fobj, sense = GRB.MINIMIZE)

		# add constraints
		model.addConstr(xp[0] + xr[0] - sp[0] == D[0])
		model.addConstrs(sp[i-1] + xp[i] + xr[i] - sp[i] == D[i] for i in range(N) if i > 0 )

		model.addConstr(R[0] - xr[0] - sr[0] == 0)
		model.addConstrs(sr[i-1] + R[i] - xr[i] - sr[i] == 0 for i in range(N) if i > 0)

		# without capacity
		model.addConstrs(xp[i] - yp[i]*SD[i][N-1] <= 0 for i in range(N))
		model.addConstrs(xr[i] - yr[i]*min(SR[0][i],SD[i][N-1]) <= 0 for i in range(N))

		# with capacity
 		#model.addConstrs(xp[i] - yp[i]*min(SD[i][N-1],C) <= 0 for i in range(N))
		#model.addConstrs(xr[i] - yr[i]*min(SR[0][i],SD[i][N-1],C) <= 0 for i in range(N))
		#model.addConstrs(xp[i] + xr[i] <= C for i in range(N))

		for i in range(2,N):
			for l in range(i,N):
					con = 0
					for k in range(i,l+1):
						con += SD[k][l]*(yp[k]+yr[k])
					model.addConstr(sp[i-1] + con >= SD[i][l])

		for i in range(1,N):
			for l in range(i,N):
					con = 0
					for k in range(i,l+1):
						con += SR[i][k]*yr[k]
					model.addConstr(sr[l] + con >= SR[i][l])

		# export .lp .mps
		#model.write(f"file_format/{file_name}+"_model.lp")

		# set parameters 
		#model.setParam("LogFile", "gurobi_lrelax.log")
		#model.setParam("LogToConsole", 0)
		model.setParam('OutputFlag', 0)
		#model.Params.OutputFlag = -1		
		#model.Params.LogFile("gurobi_log.log")
		
		model.Params.TimeLimit = MAX_CPU_TIME
		model.Params.MIPGap = EPSILON
		model.Params.Threads = 1
		#model.Params.Cuts = -1
		#model.Params.Presolve = -1
		#model.Params.method = 0 #-1=automatic, 0=primal, 1=dual , 2=barrier
		#model.Params.NodeMethod = 1 #-1=automatic, 0=primal, 1=dual , 2=barrier

		# optimize model
		model.optimize()
		
		tmp = 0
		if model.status == GRB.OPTIMAL:
			tmp = 1

	except gp.GurobiError as e:
		print('Error code ' + str(e.errno) + ': ' + str(e))
	
	objval = model.ObjVal
	objbound = model.ObjBound
	mipgap = model.MIPGap
	runtime = model.Runtime
	nodecount = model.NodeCount

	return objval, objbound, mipgap, runtime, nodecount, tmp

In [248]:
def gera_particoes(N, tam_part, num_part_fix):
	
	tam_janela = tam_part - num_part_fix
	subset = []
	 
	for i in range(0, N, tam_janela):
		if i + tam_part > N:
			subset.append([k for k in range(i,N)])
		else:
			subset.append([k for k in range(i,i+tam_part)])

	return subset

In [249]:
def upper_bound(N, PP, PR, FP, FR, HR, HP, D, R, SD, SR, C):
	
	yp_val = np.zeros(N)
	yr_val = np.zeros(N)

	# parameters rf
	tam_partp_rf = 5
	num_fixp_rf = 2
	tam_partr_rf = 6
	num_fixr_rf = 3
	
	# parameters fo
	tam_partp_fo = 7
	num_fixp_fo = 2
	tam_partr_fo = 8
	num_fixr_fo = 3

	subsetp_rf = gera_particoes(N, tam_partp_rf, num_fixp_rf)
	subsetr_rf = gera_particoes(N, tam_partr_rf, num_fixr_rf)
	subsetp_fo = gera_particoes(N, tam_partp_fo, num_fixp_fo)
	subsetr_fo = gera_particoes(N, tam_partr_fo, num_fixr_fo)
	
	start_time = time.time()
	for conjp, conjr in itertools.zip_longest(subsetp_rf,subsetr_rf): #zip
		rf_objval, yp_val, yr_val = relax_fix(conjp, conjr, yp_val, yr_val, N, PP, PR, FP, FR, HP, HR, D, R, SD, SR)
	rf_rtime = time.time() - start_time
	
	start_time = time.time()
	for conjp, conjr in itertools.zip_longest(subsetp_fo,subsetr_fo):
		fo_objval, xp_val, xr_val, yp_val, yr_val, sp_val, sr_val = fix_and_optimize(conjp, conjr, yp_val, yr_val, N, PP, PR, FP, FR, HP, HR, D, R, SD, SR)
	fo_rtime = time.time() - start_time

	return fo_objval, xp_val, yp_val, sp_val, xr_val, yr_val, sr_val
    

In [252]:
def main(file_name):
	
	N, PP, PR, FP, FR, HR, HP, D, R, C = read_instance(os.path.join(instances,file_name))

	SD = (np.zeros((N,N))).tolist()
	SR = (np.zeros((N,N))).tolist()

	for  i in range(N):
		SD[i][i] = D[i]
		SR[i][i] = R[i]
		for j in range(i+1,N):
			SD[i][j] = SD[i][j-1] + D[j]
			SR[i][j] = SR[i][j-1] + R[j]

	#N = 10

	#objval, objbound, gap, runtime, node, tmp = clsr_std_mip(N, PP, PR, FP, FR, HR, HP, D, R, SD, SR, C)
	
	#arquivo = open(os.path.join(result,'clsr_std_mip_lrelax.csv'),'a')
	
	#arquivo.write(file_name+';'
	#		   +str(round(objval,2))+';'
	#		   +str(round(objbound,2))+';'
	#		   +str(round(gap,2))+';'
	#		   +str(round(runtime,2))+';'
	#		   +str(round(node,2))+';'
	#		   +str(round(tmp,2))+'\n')
	#arquivo.close()
	# 
		
	#arquivo = open(os.path.join(result,'lsr_std_math_lrelax.txt'),'a')
	#arquivo.write(file_name+';'
	#	+str(round(rf_objval,2))+';'
	#	+str(round(rf_rtime,2))+';'
	#	+str(round(fo_objval,2))+';'
	#	+str(round(fo_rtime,2))
	#	+'\n')
	#arquivo.close()

	MAX_ITER = 10

	# The best-known upper and lower bounds
	Z_UB = np.Inf
	Z_LB = -np.Inf

	# The best-known feasible solutions
	xp_best = np.zeros(N)
	xr_best = np.zeros(N)
	yp_best = np.zeros(N)
	yr_best = np.zeros(N)
	sp_best = np.zeros(N)
	sr_best = np.zeros(N)
	
	lambdap = np.zeros(N)
	lambdar = np.zeros(N)

	gradp = np.zeros(N)

	for k in range(1,MAX_ITER):
		print(f"it = {k}")
		Z_P, xp_p, yp_p, sp_p, xr_p, yr_p, sr_p = upper_bound(N, PP, PR, FP, FR, HR, HP, D, R, SD, SR, C)
		#print(f"primal = {Z_P}")
		
		Z_D, xp_d, yp_d, sp_d, xr_d, yr_d, sr_d = lower_bound(N, PP, PR, FP, FR, HP, HR, D, R, SD, SR, lambdap)
		#print(f"dual = {Z_D}")

		# Updating the upper bound
		if Z_P < Z_UB:
			Z_UB = Z_P
			xp_best = xp_p
			yp_best = yp_p
			sp_best = sp_p
			xr_best = xr_p
			yr_best = yr_p
			sr_best = sr_p
		
		# Updating the lower bound
		if Z_D > Z_LB:
			Z_LB = Z_D

		# subgradiente
		for i in range(N):
			gradp[i] = xp_best[i] - SD[i][N-1]*yp_best[i]

		# Qgrad
		Qgrad = 0
		for i in range(N):
			Qgrad += gradp[i] * gradp[i]

		# Determining the step size and updating the multiplier
		theta = 1.0
		t = theta * (Z_UB - Z_D) / Qgrad
		print(f"Qgrad = {Qgrad}, Z_UB = {Z_UB}, Z_D = {Z_D}, t = {t}")

		for i in range(N):
			lambdap[i] = max(0,t*gradp[i])
		
		# Computing the optimality gap
		opt_gap = (Z_UB - Z_LB) / Z_UB

		if opt_gap < 0.000001:
			print("opt gap small")
			break

		if t < 0.000001:
			print("step t small")
			break

	return Z_LB, Z_UB #xp_best, yp_best, sp_best, xr_best, yr_best, sr_best

In [253]:
if __name__== "__main__" :

	for dim in [52]:
		for id in range(1,2):
			datafile = f"c{dim}_{id}.txt"
			print(f"Resolvendo instancia {datafile}")			
			Z_LB, Z_UB = main(datafile)
			print(f"Z_LB = {Z_LB}, Z_UB = {Z_UB}")

Resolvendo instancia c52_1.txt
it = 1
Qgrad = 165172235.0, Z_UB = 8706.400000000001, Z_D = 7584.2, t = 6.794120089251088e-06
it = 2
Qgrad = 165172235.0, Z_UB = 8706.400000000001, Z_D = 7584.2, t = 6.794120089251088e-06
it = 3
Qgrad = 165172235.0, Z_UB = 8706.400000000001, Z_D = 7584.2, t = 6.794120089251088e-06
it = 4
Qgrad = 165172235.0, Z_UB = 8706.400000000001, Z_D = 7584.2, t = 6.794120089251088e-06
it = 5
Qgrad = 165172235.0, Z_UB = 8706.400000000001, Z_D = 7584.2, t = 6.794120089251088e-06
it = 6
Qgrad = 165172235.0, Z_UB = 8706.400000000001, Z_D = 7584.2, t = 6.794120089251088e-06
it = 7
Qgrad = 165172235.0, Z_UB = 8706.400000000001, Z_D = 7584.2, t = 6.794120089251088e-06
it = 8
Qgrad = 165172235.0, Z_UB = 8706.400000000001, Z_D = 7584.2, t = 6.794120089251088e-06
it = 9
Qgrad = 165172235.0, Z_UB = 8706.400000000001, Z_D = 7584.2, t = 6.794120089251088e-06
Z_LB = 7584.2, Z_UB = 8706.400000000001


## Table of results

In [178]:
data=f'result/clsr_std_mip_lrelax.csv'

df = pd.DataFrame()
df = pd.read_csv(data,header=None,sep=';')

tab = pd.DataFrame()
tab = pd.concat([tab, df], ignore_index=True)
tab.columns = ['instance','objval','objbound','mipgap','time','nodes','opt']

resume = pd.DataFrame({
    'instance':f"resume",
    'objval':tab["objval"].mean(),
    'objbound':tab["objbound"].mean(),
    'mipgap':tab['mipgap'].mean(),
    'time':tab['time'].mean(),
    'nodes':tab['nodes'].mean(),
    'opt':tab['opt'].sum(),
     },index=[f"uls_mip"]
)

tab = pd.concat([tab, resume], ignore_index=True)

tab["objval"] = tab["objval"].round(2)
tab["objbound"] = tab["objbound"].round(2)
tab["mipgap"] = tab["mipgap"].round(2)
tab["time"] = tab["time"].round(2)
tab["nodes"] = tab["nodes"].round(2)
tab["opt"] = tab["opt"].round().astype('Int64')

tab_mip = tab
tab_mip


,instance,objval,objbound,mipgap,time,nodes,opt
0,c52_1.txt,2574.40,2574.40,0.0,0.06,5.00,1
1,c52_1.txt,1726.60,1726.60,0.0,0.04,8.00,1
2,c52_1.txt,1726.60,1726.60,0.0,0.05,5.00,1
3,c52_1.txt,1726.60,1726.60,0.0,0.06,7.00,1
4,c52_1.txt,1726.60,1726.60,0.0,0.05,7.00,1
5,c52_1.txt,8698.80,8689.24,0.0,60.00,9532.00,0
6,c52_1.txt,8698.80,8698.80,0.0,60.92,9715.00,1
7,c52_1.txt,8698.80,8698.80,0.0,60.23,9715.00,1
8,resume,4447.15,4445.96,0.0,22.68,3624.25,7


In [179]:
data=f'result/lsr_std_math_lrelax.txt'

df = pd.DataFrame()
df = pd.read_csv(data,header=None,sep=';')

tab = pd.DataFrame()
tab = pd.concat([tab, df], ignore_index=True)
tab.columns = ['instance','rf_objval','rf_rtime','fop_objval','fop_rtime']

resume = pd.DataFrame({
    'instance':f"resume",
    'rf_objval':tab["rf_objval"].mean(),
    'rf_rtime':tab["rf_rtime"].mean(),
    'fop_objval':tab['fop_objval'].mean(),
    'fop_rtime':tab['fop_rtime'].mean(),
     },index=[f"uls_math"]
)

tab = pd.concat([tab, resume], ignore_index=True)

tab["rf_objval"] = tab["rf_objval"].round(2)
tab["rf_rtime"] = tab["rf_rtime"].round(2)
tab["fop_objval"] = tab["fop_objval"].round(2)
tab["fop_rtime"] = tab["fop_rtime"].round(2)

tab_rf = tab
tab_rf

,instance,rf_objval,rf_rtime,fop_objval,fop_rtime
0,c52_1.txt,8915.4,1.35,8751.6,1.36
1,c52_1.txt,8915.4,1.39,8751.6,1.45
2,c52_1.txt,8915.4,0.53,8751.6,0.32
3,c52_1.txt,8915.4,0.53,8751.6,0.32
4,c52_1.txt,8915.4,0.57,8751.6,0.32
5,c52_1.txt,8915.4,0.52,8751.6,0.34
6,resume,8915.4,0.82,8751.6,0.68


In [82]:
#tab
print(
    tab_mip[['instance','objval','objbound','mipgap','time','nodes','opt']].
    to_latex(index=False,float_format="%.2f")
)

\begin{tabular}{lrrrrrr}
\toprule
instance & objval & objbound & mipgap & time & nodes & opt \\
\midrule
c52_1.txt & 2574.40 & 2574.40 & 0.00 & 0.06 & 5.00 & 1 \\
c52_1.txt & 1726.60 & 1726.60 & 0.00 & 0.04 & 8.00 & 1 \\
c52_1.txt & 1726.60 & 1726.60 & 0.00 & 0.05 & 5.00 & 1 \\
c52_1.txt & 1726.60 & 1726.60 & 0.00 & 0.06 & 7.00 & 1 \\
c52_1.txt & 1726.60 & 1726.60 & 0.00 & 0.05 & 7.00 & 1 \\
c52_1.txt & 8698.80 & 8689.24 & 0.00 & 60.00 & 9532.00 & 0 \\
c52_1.txt & 8698.80 & 8698.80 & 0.00 & 60.92 & 9715.00 & 1 \\
c52_1.txt & 8698.80 & 8698.80 & 0.00 & 60.23 & 9715.00 & 1 \\
resume & 4447.15 & 4445.96 & 0.00 & 22.68 & 3624.25 & 7 \\
\bottomrule
\end{tabular}

